# 0 · Feature: holiday day type

**Purpose.** This notebook builds the **holiday feature** for the model. For every row (date × product group) it says which kind of day the date is, and turns that into columns a linear regression can use.

**The rule, in short** (`DECISIONS.md`, 2026-11-26 and 2026-11-27):
- An **open public holiday** is every public holiday in Schleswig-Holstein **except Good Friday**. By law (Ladenöffnungszeitengesetz SH, § 4) a bakery may open on all other public holidays for five hours. Other closures are the bakery's own choice, and the bakery knows them in advance.
- Besides normal days there are four **day types**: open public holiday, Easter/Whit Sunday, Christmas Eve and New Year's Eve. They behave differently. For example, on open public holidays bread sells *less* than usual, while on New Year's Eve cakes sell about 7× as much.
- The feature is built from the **calendar only**, never from sales. So it is known long before 14:00 on the day before (canvas block 6), and it cannot leak the target.

**Input:** the prepared data from `01_load_join_clean.ipynb`. **Output:** `output/features_holidays.csv`, one row per date × product group, to be joined on `Datum` and `Warengruppe`.

**Checkpoints.** The notebook runs the same checks as `uv run pytest checks` (K14–K16, canvas block 8). If a check fails, the notebook stops with a plain message. Do not change the check; find out what went wrong.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Make the shared code (src/) and the checks (checks/) importable
ROOT = Path.cwd().parent if Path.cwd().name == "0_DataPreparation" else Path.cwd()
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "checks")]

import bakery_holidays as hol
import invariants as inv

WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
GROUPS = {1: "Bread", 2: "Rolls", 3: "Croissants", 4: "Confectionery", 5: "Cakes", 6: "Seasonal bread"}

## 1. Load the prepared data

The prepared data has one row per date and product group, only on days with sales. Each row is marked as *train* or *validation*. A date without any row means the bakery was closed, and such dates are never predicted.

In [2]:
data = pd.read_csv(ROOT / "0_DataPreparation" / "output" / "bakery_prepared.csv", parse_dates=["Datum"])
train_mask = data["split"] == "train"

pd.DataFrame({
    "rows": data.groupby("split").size(),
    "dates": data.groupby("split")["Datum"].nunique(),
    "first date": data.groupby("split")["Datum"].min().dt.date,
    "last date": data.groupby("split")["Datum"].max().dt.date,
})

,rows,dates,first date,last date
split,,,,
train,7493,1462,2013-07-01,2017-07-31
validation,1841,357,2017-08-01,2018-07-31


## 2. Day type of each date

`day_type` looks up each date in the calendar:
- public holidays of Schleswig-Holstein from the `holidays` package, without Good Friday → `open_holiday`
- Easter Sunday and Whit Sunday → `easter_whit_sunday`; 24 December → `christmas_eve`; 31 December → `new_years_eve`
- everything else → `normal` (the reference)

The table counts **dates**, not rows, per day type.

In [3]:
data["day_type"] = hol.day_type(data["Datum"]).to_numpy()

days = data.drop_duplicates("Datum")
days.groupby(["day_type", "split"]).size().unstack("split").reindex(hol.DAY_TYPES)

split,train,validation
day_type,,
normal,1429,350
open_holiday,17,3
easter_whit_sunday,8,2
christmas_eve,4,1
new_years_eve,4,1


**Checkpoint "features" (K15).**
- Do the day types fall exactly on the known dates? The dates are written out by hand in `checks/invariants.py`, independent of the `holidays` package.
- Are the counts per split the decided ones (training 17 / 8 / 4 / 4, validation 3 / 2 / 1 / 1)?
- Is Good Friday never an open public holiday?

In [4]:
inv.check_holiday_day_types(data["day_type"], data, hol.day_type)
print("K15 passed")

K15 passed


### Which weekdays do they fall on?

Weekdays matter here. A holiday on a Monday replaces a normal Monday, and a special day on a Sunday replaces a normal Sunday.

In [5]:
special = days[days["day_type"] != "normal"].copy()
special["weekday"] = pd.Categorical(special["Datum"].dt.day_name(), categories=WEEKDAYS, ordered=True)
(special.groupby(["day_type", "weekday", "split"], observed=True).size()
        .unstack("split", fill_value=0)
        .reindex(hol.DAY_TYPES[1:], level=0))

split                         train  validation
day_type           weekday                     
open_holiday       Monday         9           1
                   Tuesday        0           1
                   Thursday       5           1
                   Friday         1           0
                   Saturday       1           0
                   Sunday         1           0
easter_whit_sunday Sunday         8           2
christmas_eve      Tuesday        1           0
                   Wednesday      1           0
                   Thursday       1           0
                   Saturday       1           0
                   Sunday         0           1
new_years_eve      Tuesday        1           0
                   Wednesday      1           0
                   Thursday       1           0
                   Saturday       1           0
                   Sunday         0           1

**What to notice:**
- In training, open public holidays fall almost only on Mondays (Easter and Whit Monday) and Thursdays (Ascension Day). Only German Unity Day moves through the week.
- Christmas Eve and New Year's Eve fell on a **Sunday** in the validation year (2017). That never happens in training. In the test year (2018/19) both are Mondays. So the model has to combine the weekday effect with the day-type effect for combinations it has never seen.
- Christmas Eve and New Year's Eve have only 4 training days each, so their effects will be uncertain.

## 3. Interaction with the product group

A linear regression gives each column **one** effect for all rows. But the holiday effect differs by product group, even in direction: on open public holidays bread sells −41 % and confectionery +98 % (H3). So we build **interaction columns**. For example, `open_holiday_g1` is 1 only on an open public holiday **and** in group 1 (bread). This way each group gets its own effect per day type.

Two rules:
- **No extra column per day type.** It would be the sum of its interaction columns, so the same information would be in the model twice.
- **Combinations that never occur in training are left out.** For example, seasonal bread is never sold on an open public holiday, so that column would always be 0. Which combinations are kept is learned from the **training period only**.

In [6]:
columns = hol.interaction_columns(data[train_mask])
print(f"{len(columns)} interaction columns:")
for t in hol.DAY_TYPES[1:]:
    groups = [GROUPS[int(c.rsplit("_g", 1)[1])] for c in columns if c.rsplit("_g", 1)[0] == t]
    print(f"  {t:<20} {len(groups)}: {', '.join(groups)}")

22 interaction columns:
  open_holiday         5: Bread, Rolls, Croissants, Confectionery, Cakes
  easter_whit_sunday   5: Bread, Rolls, Croissants, Confectionery, Cakes
  christmas_eve        6: Bread, Rolls, Croissants, Confectionery, Cakes, Seasonal bread
  new_years_eve        6: Bread, Rolls, Croissants, Confectionery, Cakes, Seasonal bread


In [7]:
features = hol.holiday_features(data, columns)

# Example: a normal Monday, Easter Monday 2017 and New Year's Eve 2016, for bread, rolls and cakes
example_days = pd.to_datetime(["2017-04-10", "2017-04-17", "2016-12-31"])
shown = ["open_holiday_g1", "open_holiday_g2", "open_holiday_g5",
         "new_years_eve_g1", "new_years_eve_g2", "new_years_eve_g5"]
example = features[features["Datum"].isin(example_days) & features["Warengruppe"].isin([1, 2, 5])]
example.assign(Umsatz=data.loc[example.index, "Umsatz"].round(0))[["Datum", "Warengruppe", "day_type", "Umsatz", *shown]]

,Datum,Warengruppe,day_type,Umsatz,open_holiday_g1,open_holiday_g2,open_holiday_g5,new_years_eve_g1,new_years_eve_g2,new_years_eve_g5
1253,2016-12-31,1,new_years_eve,289.0,0,0,0,1,0,0
1352,2017-04-10,1,normal,157.0,0,0,0,0,0,0
1358,2017-04-17,1,open_holiday,23.0,1,0,0,0,0,0
3072,2016-12-31,2,new_years_eve,570.0,0,0,0,0,1,0
3171,2017-04-10,2,normal,372.0,0,0,0,0,0,0
3177,2017-04-17,2,open_holiday,536.0,0,1,0,0,0,0
8476,2016-12-31,5,new_years_eve,1705.0,0,0,0,0,0,1
8575,2017-04-10,5,normal,280.0,0,0,0,0,0,0
8581,2017-04-17,5,open_holiday,240.0,0,0,1,0,0,0


**How to read it:** on the normal Monday all columns are 0. On Easter Monday and on New Year's Eve each row has exactly one 1, in the column of its own group. The regression learns one coefficient per column, e.g. "the extra revenue of rolls on an open public holiday".

**Checkpoint "features" (K14, K16).**
- **K14:** do the features stay identical when revenue is shuffled or removed? If so, they use only the date and the group.
- **K16:** is each column 1 exactly on its own day type and in its own group? Is there at most one 1 per row, and is no column always 0 in training?

In [8]:
inv.check_holiday_feature_date_only(lambda d: hol.holiday_features(d, columns), data.drop(columns="day_type"))
inv.check_holiday_interactions(features, columns, train_mask)
print("K14 and K16 passed")

K14 and K16 passed


## 4. Save the feature table

The table has one row per row of the prepared data and is keyed by `Datum` and `Warengruppe`, so it can be joined with the other features before modelling. Revenue is not included.

In [9]:
out_path = ROOT / "0_DataPreparation" / "output" / "features_holidays.csv"
assert len(features) == len(data), "Feature table has a different number of rows than the prepared data"
assert not features.duplicated(["Datum", "Warengruppe"]).any(), "Key (Datum, Warengruppe) is not unique"
features.to_csv(out_path, index=False, date_format="%Y-%m-%d")
print(f"Saved {len(features):,} rows x {features.shape[1]} columns to {out_path.relative_to(ROOT).as_posix()}")

Saved 9,334 rows x 25 columns to 0_DataPreparation/output/features_holidays.csv


## Summary and limits

- **Built:** `day_type` (5 values) and 22 interaction columns day type × product group, from the calendar only. Checks K14–K16 pass.
- **How sure:** the direction of the holiday effects is clear (H3), their size is not. Christmas Eve and New Year's Eve rest on 4 training days each, and the validation year has only 1 case each. So they can only be judged together with the other special days.
- **Not covered yet:**
  - The test year (2018-08 – 2019-07) needs the same table once the Kaggle test file is loaded. `day_type` works for any date. When that happens, check whether Reformation Day (2018-10-31) and 1 May 2019 have rows. Both count as open public holidays, but the model has never seen them open.
  - Days around holidays and bridge days are possible later experiments.
  - The law text was read in its 2026 version. Whether § 4 read the same in 2013–2019 is unchecked.